# ⚡ LangGraph Approval Hub — Interactive Demo

This notebook walks you through the **full approval loop** from a Python agent to a human decision in the dashboard.

### What you'll see
1. An approval request sent from Python → appears live in the dashboard
2. You approve or reject it in the browser
3. Python receives the decision and continues

### What you need
- Your **hub URL** (e.g. `https://your-hub.vercel.app`)
- Your **API secret token** (from Vercel → Project → Settings → Environment Variables → `API_SECRET_TOKEN`)
- Python 3.9+

---

## Step 1 — Install the SDK

Run the cell below. If you've already installed it, this is instant.

In [ ]:
%pip install git+https://github.com/suryamr2002/langgraph-approval-hub.git#subdirectory=sdk --quiet

## Step 2 — Configure your hub

Enter your hub URL and API token below. The token is hidden as you type.

In [ ]:
import getpass

HUB_URL = input("Hub URL (e.g. https://your-hub.vercel.app): ").strip().rstrip("/")
API_TOKEN = getpass.getpass("API Secret Token: ")

print(f"\n✅ Hub configured: {HUB_URL}")

## Step 3 — Your first approval request

This simulates a **Finance Agent** asking a human to approve a refund.

**What will happen:**
1. Run the cell → the request is sent to your hub
2. Open your hub in the browser — you'll see it appear under **Pending Approvals**
3. Click **Approve** or **Reject**
4. This cell prints the decision and continues

> ⚠️ The cell blocks until you decide. Keep the browser open in another tab.

In [ ]:
from langgraph_approval_hub import request_approval

YOUR_EMAIL = input("Your email (you'll see the request here): ").strip()

print("\n⏳ Waiting for your decision in the dashboard...")
print(f"   Open → {HUB_URL}\n")

decision = request_approval(
    hub_url=HUB_URL,
    api_token=API_TOKEN,
    agent_name="Finance Agent",
    action_description="Process $4,200 refund batch for 12 customers",
    assignee=YOUR_EMAIL,
    assignee_type="person",
    timeout_minutes=10,
)

if decision == "approved":
    print("✅ Approved — agent would now process the refunds.")
else:
    print("❌ Rejected — agent stopped. No action taken.")

## Step 4 — Route to a team

Instead of a single person, route to a **team** — every team member gets notified, first to respond wins.

**Before running:** Go to your hub → **Settings → Teams** and create a team (e.g. `finance-team`) with your email as a member.

In [ ]:
TEAM_NAME = input("Team name from Settings → Teams (e.g. finance-team): ").strip()

print("\n⏳ Waiting for team decision...")
print(f"   Open → {HUB_URL}\n")

decision = request_approval(
    hub_url=HUB_URL,
    api_token=API_TOKEN,
    agent_name="Data Pipeline Agent",
    action_description="Drop and recreate the events_staging table (30 min downtime)",
    assignee=TEAM_NAME,
    assignee_type="team",
    timeout_minutes=10,
)

print(f"\nDecision: {decision}")
if decision == "approved":
    print("✅ Team approved — pipeline would now run.")
else:
    print("❌ Team rejected — pipeline aborted.")

## Step 5 — Escalation

Set a timeout. If the assignee doesn't respond in time, the hub automatically:
- Marks the request as **Escalated** on the dashboard
- Notifies the escalation target

Here we use a **1-minute timeout** so you can see it escalate quickly.

In [ ]:
ESCALATE_TO = input("Escalation email (can be the same as yours): ").strip()

print("\n⏳ Waiting... (don't approve this one for 1 minute to see escalation)")
print(f"   Open → {HUB_URL}\n")

try:
    decision = request_approval(
        hub_url=HUB_URL,
        api_token=API_TOKEN,
        agent_name="Billing Agent",
        action_description="Issue $12,000 credit to enterprise account ACME-001",
        assignee=YOUR_EMAIL,
        assignee_type="person",
        escalate_to=ESCALATE_TO,
        timeout_minutes=1,
    )
    print(f"Decision received: {decision}")
except TimeoutError:
    print("⏰ Timed out — escalation triggered. Check the dashboard (status = Escalated).")
    print("   Agent took the safe path: no action.")

## Step 6 — Agent reasoning

Pass `agent_reasoning` to give the approver full context. It appears on the approval detail page in the dashboard — the approver sees *why* the agent wants to do this before deciding.

In [ ]:
print("⏳ Waiting for your decision...")
print(f"   Open → {HUB_URL}")
print("   Click the request to see the agent's reasoning.\n")

decision = request_approval(
    hub_url=HUB_URL,
    api_token=API_TOKEN,
    agent_name="Customer Service Agent",
    action_description="Issue full refund of $340 to customer #A-4821",
    assignee=YOUR_EMAIL,
    assignee_type="person",
    agent_reasoning="""
1. Customer purchased on 2024-03-01, within the 30-day return window.
2. Item returned in original packaging — no restocking fee applies.
3. Customer account has no prior refund requests in the last 12 months.
4. Refund amount ($340) matches the original charge exactly.
5. Confidence: high. Recommend approval.
""",
    timeout_minutes=10,
)

print(f"\nDecision: {decision}")

## Step 7 — Async / batch mode

Use  +  when you don't want to block.  
Submit many tasks up front, let humans review in the dashboard, then check results whenever you're ready.

This is ideal for **end-of-day batch workflows** or **fire-and-forget tasks** where the agent can do other work while waiting.

In [ ]:
from langgraph_approval_hub import submit_approval, get_decision, get_pending

# Submit approvals without blocking -- returns an ID immediately
print("Submitting batch of tasks...")

id1 = submit_approval(
    hub_url=HUB_URL,
    api_token=API_TOKEN,
    agent_name="Content Agent",
    action_description="Publish Q2 earnings blog post",
    assignee=YOUR_EMAIL,
    assignee_type="person",
    timeout_minutes=60,
)

id2 = submit_approval(
    hub_url=HUB_URL,
    api_token=API_TOKEN,
    agent_name="DevOps Agent",
    action_description="Rotate production database credentials",
    assignee=YOUR_EMAIL,
    assignee_type="person",
    timeout_minutes=60,
)

print("Submitted 2 tasks (not blocking)")
print("  Task 1 ID:", id1)
print("  Task 2 ID:", id2)
print()
print("Go approve/reject them in the dashboard:", HUB_URL)
print("Then run the next cell to check results.")

### Check results later

Run this cell any time after deciding in the dashboard — no need to wait for a specific moment.

In [ ]:
# Check what is still pending
pending = get_pending(hub_url=HUB_URL, api_token=API_TOKEN)
print("Still pending:", len(pending), "task(s)")
for t in pending:
    name = t.get("agent_name", "?")
    desc = t.get("action_description", "?")
    print("  -", name, "--", desc)

# Check individual decisions
s1 = get_decision(hub_url=HUB_URL, api_token=API_TOKEN, approval_id=id1)
s2 = get_decision(hub_url=HUB_URL, api_token=API_TOKEN, approval_id=id2)

print()
print("Task 1 (Publish blog post):     ", s1)
print("Task 2 (Rotate DB credentials): ", s2)

## Step 8 — Audit log

Every decision is recorded. Let's pull the full audit log directly from the API.

In [ ]:
import requests

response = requests.get(
    HUB_URL + "/api/audit",
    headers={"Authorization": "Bearer " + API_TOKEN},
)

data = response.json()
records = data.get("records") or []
total = data.get("total", len(records))

print("Total decisions recorded:", total)
print()
for entry in records[:5]:
    status = entry.get("status", "?").upper().ljust(8)
    agent  = entry.get("agent_name", "?")
    action = entry.get("action_description", "")[:60]
    print(" [", status, "]", agent, "--", action)

---

## Deploy your own hub

Everything above ran against a hosted hub. To deploy your own:

1. Click **Deploy to Vercel** on [github.com/suryamr2002/langgraph-approval-hub](https://github.com/suryamr2002/langgraph-approval-hub)
2. Set env vars: , , , , 
3. Optional: add  for email notifications,  for Slack
4. Point  in this notebook to your new URL

Full docs and SDK reference: [github.com/suryamr2002/langgraph-approval-hub](https://github.com/suryamr2002/langgraph-approval-hub#readme)